# 🧠 RAG Vector Store — Deep Dive & Visualisation

**How Sobhan's personal knowledge base goes from Markdown files to a searchable vector database.**

---

## What you will explore in this notebook

We have 8 Markdown files about Sobhan Dutta — his career history, UX/AI expertise, and education.
By the end of this notebook you will have:

1. Loaded and measured the raw documents (characters, tokens, files)
2. Split them into overlapping **chunks** suitable for retrieval
3. Converted every chunk into a high-dimensional **embedding vector** using OpenAI
4. Stored those vectors in a persistent **ChromaDB** vector store
5. **Visualised** the vector space in 2-D and 3-D using t-SNE
6. Run live **queries** and watched where they land in vector space

---

## The three parts

| Part | Topic | Key concept |
|---|---|---|
| **A** | Documents → Chunks | Why we split and how overlap prevents information loss |
| **B** | Chunks → Vectors | What an embedding is and why dimensions matter |
| **C** | Visualise the space | t-SNE, cluster structure, query placement, live retrieval |

> **Prerequisites:** `OPENAI_API_KEY` in a `.env` file — needed for embeddings.
> Run `python data/ingest_kb.py` first to build the vector store on disk.
> jupyter notebook rag_visualization.ipynb to open notebook

In [1]:
import os, glob
import numpy as np
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

from openai import OpenAI
from chromadb import PersistentClient

from sklearn.manifold import TSNE
import plotly.graph_objects as go

import tiktoken

load_dotenv(override=True)

openai = OpenAI()
api_key = os.getenv("OPENAI_API_KEY", "")
print(f"OpenAI API Key found — starts with {api_key[:8]}..." if api_key else "❌ OPENAI_API_KEY not set")

# Paths — relative to this notebook (which lives in the agentic/ root)
KB_PATH          = Path("knowledge_base")
VECTOR_STORE_PATH = str(Path("vector_store"))
COLLECTION_NAME  = "sobhan_knowledge_base"
EMBEDDING_MODEL  = "text-embedding-3-small"   # must match data/ingest_kb.py

print(f"\nKnowledge base : {KB_PATH.resolve()}")
print(f"Vector store   : {VECTOR_STORE_PATH}")

OpenAI API Key found — starts with sk-proj-...

Knowledge base : /Users/sobhandutta/projects/full-llm-assistant/knowledge_base
Vector store   : vector_store


---
## Part A — Documents → Chunks

### Step 1: Measure the knowledge base

Before touching any LLM, let's measure the raw data:
- **Files** — how many documents we have and how they're organised
- **Characters** — raw size on disk
- **Tokens** — what the LLM *actually* processes (roughly `chars / 4` for English)

This tells us: could we just dump everything into one prompt?
Spoiler: technically yes for 8 files — but RAG is still the right pattern for accuracy, cost, and scalability.

In [2]:
# Load every .md file and gather stats
documents = []
for md_file in sorted(KB_PATH.rglob("*.md")):
    category = md_file.parent.name   # "career", "expertise", or "education"
    text     = md_file.read_text(encoding="utf-8")
    documents.append({"path": md_file, "category": category,
                       "filename": md_file.stem, "text": text})

# Print a summary tree
print(f"{'='*50}")
print(f"  Knowledge Base: {len(documents)} documents")
print(f"{'='*50}")
category_counts = Counter(d["category"] for d in documents)
for cat, count in sorted(category_counts.items()):
    cat_docs = [d["filename"] for d in documents if d["category"] == cat]
    print(f"\n  📁 {cat}/  ({count} files)")
    for name in cat_docs:
        print(f"     └─ {name}.md")

# Measure total size
all_text = "\n\n".join(d["text"] for d in documents)
print(f"\n{'='*50}")
print(f"  Total characters : {len(all_text):>8,}")
print(f"  Average per file : {len(all_text)/len(documents):>8,.0f} chars")

  Knowledge Base: 9 documents

  📁 career/  (4 files)
     └─ ataya.md
     └─ early_career.md
     └─ elisity.md
     └─ nuance.md

  📁 education/  (1 files)
     └─ background.md

  📁 expertise/  (3 files)
     └─ frontend_engineering.md
     └─ leadership.md
     └─ ux_design_philosophy.md

  📁 youtube/  (1 files)
     └─ youtube.md

  Total characters :   25,400
  Average per file :    2,822 chars


In [3]:
# Token count using the tokeniser claude-sonnet-4-6 / gpt-4o share (cl100k_base)
encoding    = tiktoken.get_encoding("cl100k_base")
token_count = len(encoding.encode(all_text))

print(f"Total tokens in knowledge base : {token_count:,}")
print(f"Chars-per-token ratio          : {len(all_text)/token_count:.2f}  (English ≈ 4)")

# Cost and context comparison
claude_context = 200_000
gpt4_context   = 128_000
cost_full      = token_count / 1_000_000 * 0.003   # claude-haiku input price

print(f"\nFits in Claude context ({claude_context:,} tokens)?  "
      + ("✅ YES" if token_count < claude_context else "❌ NO"))
print(f"Fits in GPT-4o context ({gpt4_context:,} tokens)?   "
      + ("✅ YES" if token_count < gpt4_context else "❌ NO"))
print(f"\nCost to send ALL docs every query : ~${cost_full:.4f} (haiku input pricing)")

rag_tokens = 5 * 150  # top-5 chunks × ~150 tokens each
cost_rag   = rag_tokens / 1_000_000 * 0.003
print(f"Cost with RAG (top-5 chunks)      : ~${cost_rag:.6f}  ({cost_full/cost_rag:.0f}× cheaper)")

Total tokens in knowledge base : 5,230
Chars-per-token ratio          : 4.86  (English ≈ 4)

Fits in Claude context (200,000 tokens)?  ✅ YES
Fits in GPT-4o context (128,000 tokens)?   ✅ YES

Cost to send ALL docs every query : ~$0.0000 (haiku input pricing)
Cost with RAG (top-5 chunks)      : ~$0.000002  (7× cheaper)


### Step 2: Split documents into chunks

**Why chunk at all?**

Each document becomes a single vector if stored whole. A long document about Sobhan's career
would produce one vector that "averages" all its content — making it hard to match a specific
question like "What did Sobhan achieve at Elisity?".

Smaller, focused chunks = more precise retrieval.

**Why overlap?**

```
Doc text: |──────────────────────────────────────────────────────|

Chunk 1:  |────── 600 ──────|
Chunk 2:              |── 100 overlap ──|────── 600 ──────|
Chunk 3:                                        |── 100 ──|── 600 ──|
```

A sentence near a chunk boundary appears in **two** chunks. Without overlap, it might be
split and never fully retrieved. Overlap trades a little storage for significantly better retrieval.

In [4]:
CHUNK_SIZE    = 600   # characters per chunk (matches data/ingest_kb.py)
CHUNK_OVERLAP = 100   # overlap between adjacent chunks

def chunk_text(text, chunk_size, overlap):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += chunk_size - overlap
    return chunks

# Chunk every document and attach metadata
all_chunks = []
for doc in documents:
    for i, chunk_str in enumerate(chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)):
        if len(chunk_str.strip()) >= 50:   # skip tiny trailing fragments
            all_chunks.append({
                "text":     chunk_str,
                "source":   doc["filename"],
                "category": doc["category"],
                "chunk_index": i,
            })

print(f"Documents   : {len(documents)}")
print(f"Chunks      : {len(all_chunks)}")
print(f"Avg per doc : {len(all_chunks)/len(documents):.1f}")
print(f"\nChunk 0 preview ({len(all_chunks[0]['text'])} chars):")
print(f"  Source   : {all_chunks[0]['source']} ({all_chunks[0]['category']})")
print(f"  Content  : {all_chunks[0]['text'][:200].strip()}...")

Documents   : 9
Chunks      : 53
Avg per doc : 5.9

Chunk 0 preview (600 chars):
  Source   : ataya (career)
  Content  : # Director of Engineering (UX & UI) — Atayalan Inc

## Role Overview
Sobhan Dutta joined Atayalan Inc in January 2022 as the founding U.S. hire and Director of Engineering (UX & UI), based in San Jose...


In [5]:
chunk_lengths = [len(c["text"]) for c in all_chunks]

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=chunk_lengths, nbinsx=30,
    marker_color="#6366f1", opacity=0.85, name="Chunk size",
))
fig.add_vline(
    x=np.mean(chunk_lengths), line_dash="dash", line_color="orange",
    annotation_text=f" Mean: {np.mean(chunk_lengths):.0f} chars",
    annotation_position="top right",
)
fig.update_layout(
    title="Distribution of Chunk Sizes (characters)",
    xaxis_title="Characters per chunk", yaxis_title="Number of chunks",
    width=750, height=400, template="plotly_dark",
)
fig.show()
print(f"Min: {min(chunk_lengths)}  Max: {max(chunk_lengths)}  Mean: {np.mean(chunk_lengths):.0f}  Median: {np.median(chunk_lengths):.0f}")

Min: 128  Max: 600  Mean: 562  Median: 600


In [6]:
CATEGORY_COLORS = {"career": "#3B82F6", "expertise": "#10B981", "education": "#F59E0B", "youtube": "#F5000B"}
CATEGORIES      = list(CATEGORY_COLORS.keys())

chunk_type_counts = Counter(c["category"] for c in all_chunks)
fig = go.Figure(go.Bar(
    x=list(chunk_type_counts.keys()),
    y=list(chunk_type_counts.values()),
    marker_color=[CATEGORY_COLORS[k] for k in chunk_type_counts.keys()],
    text=list(chunk_type_counts.values()), textposition="outside",
))
fig.update_layout(
    title="Chunks per Category",
    xaxis_title="Category", yaxis_title="Number of chunks",
    width=550, height=400, template="plotly_dark",
)
fig.show()

---
## Part B — Chunks → Vectors (Embeddings)

### What is an embedding?

An embedding converts text into a list of numbers that encodes its **semantic meaning**.
Texts with similar meaning produce numerically similar (nearby) vectors.

```
"Sobhan led the UX team at Ataya"          → [0.12, -0.34, 0.89, ...]  ← 1,536 numbers
"Who was the director of design at Ataya?" → [0.11, -0.30, 0.91, ...]  ← very close!
"The price of apples in September"         → [-0.78, 0.22, -0.15, ...]  ← far away
```

**Why 1,536 dimensions?**
`text-embedding-3-small` outputs 1,536-dimensional vectors. More dimensions = finer-grained
semantic distinctions. This model is a good balance of quality vs cost.

| Model | Dimensions | Cost per 1M tokens |
|---|---|---|
| `all-MiniLM-L6-v2` (HuggingFace) | 384 | Free (local) |
| `text-embedding-3-small` (OpenAI) | 1,536 | ~$0.02 |
| `text-embedding-3-large` (OpenAI) | 3,072 | ~$0.13 |

We use `text-embedding-3-small` — the same model used in `data/ingest_kb.py` and `agents/knowledge_base_agent.py`.

In [7]:
# Connect to the pre-built ChromaDB vector store (built by data/ingest_kb.py)
chroma     = PersistentClient(path=VECTOR_STORE_PATH)
collection = chroma.get_collection(COLLECTION_NAME)

count            = collection.count()
sample           = collection.get(limit=1, include=["embeddings"])
sample_embedding = sample["embeddings"][0]
dimensions       = len(sample_embedding)

print(f"✅ Connected to ChromaDB collection: '{COLLECTION_NAME}'")
print(f"\nVectors stored  : {count:,}")
print(f"Dimensions each : {dimensions:,}")
print(f"Total floats    : {count * dimensions:,}  (each float = 4 bytes)")
print(f"Estimated size  : {count * dimensions * 4 / 1024:.1f} KB on disk")

✅ Connected to ChromaDB collection: 'sobhan_knowledge_base'

Vectors stored  : 53
Dimensions each : 1,536
Total floats    : 81,408  (each float = 4 bytes)
Estimated size  : 318.0 KB on disk


In [8]:
# What does a raw embedding vector look like?
result = collection.get(limit=1, include=["embeddings", "documents", "metadatas"])
vec    = np.array(result["embeddings"][0])

print(f"Source   : {result['metadatas'][0]['source']} ({result['metadatas'][0]['category']})")
print(f"Content  : {result['documents'][0][:120].strip()}...")
print(f"\nFirst 15 of {len(vec)} values:")
print(np.round(vec[:15], 4))
print(f"\nVector statistics:")
print(f"  Min  : {vec.min():.4f}")
print(f"  Max  : {vec.max():.4f}")
print(f"  Mean : {vec.mean():.6f}  (≈ 0 — vectors are centred)")
print(f"  Norm : {np.linalg.norm(vec):.4f}  (unit-normalised → always ≈ 1.0)")

Source   : ataya (career)
Content  : # Director of Engineering (UX & UI) — Atayalan Inc

## Role Overview
Sobhan Dutta joined Atayalan Inc in January 2022 as...

First 15 of 1536 values:
[-0.0312 -0.0311 -0.0218  0.0453 -0.0048 -0.0264 -0.0254  0.0548 -0.0021
 -0.034   0.0228 -0.076  -0.0117 -0.0339  0.0158]

Vector statistics:
  Min  : -0.0913
  Max  : 0.0839
  Mean : 0.000772  (≈ 0 — vectors are centred)
  Norm : 1.0003  (unit-normalised → always ≈ 1.0)


In [9]:
# Cosine similarity demo — the maths behind semantic search
# cosine_similarity(a, b) = dot(a, b) / (|a| × |b|)
# For unit-normalised vectors this simplifies to just: dot(a, b)

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

texts = [
    "What is Sobhan's design philosophy?",
    "Sobhan believes designing a solution requires refinement until it reaches the optimal point — a balance between familiar and unique.",
    "The weather in San Francisco is often foggy in summer.",
]

print("Embedding 3 texts with text-embedding-3-small...")
vecs = [
    np.array(openai.embeddings.create(model=EMBEDDING_MODEL, input=[t]).data[0].embedding)
    for t in texts
]

print("\nCosine similarity (1.0 = identical meaning, 0.0 = unrelated):\n")
labels = ["Q: design philosophy?", "A: design philosophy answer", "Unrelated: SF weather"]
for i, la in enumerate(labels):
    sims = [f"{cosine_similarity(vecs[i], vecs[j]):>6.3f}" for j in range(3)]
    print(f"  {la:<35} {' | '.join(sims)}")

print("\n→ Q and A are close (similar meaning) — this is why semantic search works!")
print("→ The weather sentence is far from both — it would not be retrieved.")

Embedding 3 texts with text-embedding-3-small...

Cosine similarity (1.0 = identical meaning, 0.0 = unrelated):

  Q: design philosophy?                1.000 |  0.649 |  0.106
  A: design philosophy answer          0.649 |  1.000 |  0.100
  Unrelated: SF weather                0.106 |  0.100 |  1.000

→ Q and A are close (similar meaning) — this is why semantic search works!
→ The weather sentence is far from both — it would not be retrieved.


---
## Part C — Visualise the Vector Space

### Why visualise?

Visualisation reveals the *structure* of your knowledge base:
- Do documents of the same category cluster together?
- Are there surprising overlaps between career, expertise, and education?
- Where does a user query land relative to the documents?

### The problem: 1,536 dimensions → 2 or 3

We cannot plot 1,536-dimensional space. We use **t-SNE** to compress it.

> **t-SNE (t-distributed Stochastic Neighbor Embedding)** preserves *local structure*:
> points that are **close** in the original 1,536-D space stay **close** in 2-D/3-D.
> It is not a linear projection — distances between distant clusters are not meaningful.
> Use it to see *which clusters exist*, not to measure inter-cluster distances.

```
1,536 dimensions ──[t-SNE]──► 2 dimensions  (fast, great for spotting clusters)
1,536 dimensions ──[t-SNE]──► 3 dimensions  (richer structure, interactive rotation)
```

In [10]:
# Fetch ALL vectors + metadata from ChromaDB
result    = collection.get(include=["embeddings", "documents", "metadatas"])
vectors   = np.array(result["embeddings"])    # shape: (43, 1536)
doc_texts = result["documents"]
metadatas = result["metadatas"]
categories= [m["category"] for m in metadatas]
sources   = [m["source"]   for m in metadatas]
colors    = [CATEGORY_COLORS.get(c, "#6B7280") for c in categories]

# Hover text: category + source + first 150 chars of content
hover_texts = [
    f"<b>{cat.title()} — {src}</b><br>{txt[:150]}..."
    for cat, src, txt in zip(categories, sources, doc_texts)
]

print(f"Fetched {vectors.shape[0]} vectors, each with {vectors.shape[1]} dimensions")
print(f"\nCategory breakdown:")
for cat, count in Counter(categories).items():
    print(f"  {CATEGORY_COLORS[cat]}  {cat:<12} {count} chunks")

Fetched 53 vectors, each with 1536 dimensions

Category breakdown:
  #3B82F6  career       20 chunks
  #F59E0B  education    5 chunks
  #10B981  expertise    18 chunks
  #F5000B  youtube      10 chunks


In [11]:
# ── 2D t-SNE ──────────────────────────────────────────────────────────────────

# Sanity check before running t-SNE — diagnoses the most common failure causes
print(f"vectors type  : {type(vectors)}")
print(f"vectors shape : {vectors.shape}")
print(f"vectors dtype : {vectors.dtype}")
print(f"NaN present   : {np.isnan(vectors).any()}")
print(f"Inf present   : {np.isinf(vectors).any()}")
print(f"\nCategories in store   : {sorted(set(categories))}")
print(f"Categories in COLORS  : {CATEGORIES}")
missing = [c for c in CATEGORIES if c not in set(categories)]
if missing:
    print(f"\n⚠️  Categories in CATEGORY_COLORS but NOT in vector store: {missing}")
    print("   → Run `python data/ingest_kb.py` to rebuild the store, then re-run all cells.")

# t-SNE requires perplexity < n_samples
n_samples = vectors.shape[0]
perplexity = min(10, n_samples - 1)   # auto-clamp so it never fails
print(f"\nn_samples={n_samples}, using perplexity={perplexity}")

print("\nRunning t-SNE (2D)... this takes a few seconds")
tsne_2d    = TSNE(n_components=2, random_state=42, perplexity=perplexity)
reduced_2d = tsne_2d.fit_transform(vectors.astype(np.float32))   # float32 is faster

# Only plot categories that actually exist in the store
active_categories = [c for c in CATEGORIES if c in set(categories)]

fig = go.Figure()
for category in active_categories:
    mask = np.array([c == category for c in categories])
    fig.add_trace(go.Scatter(
        x=reduced_2d[mask, 0], y=reduced_2d[mask, 1],
        mode="markers",
        name=category.title(),
        marker=dict(size=10, color=CATEGORY_COLORS[category], opacity=0.85,
                    line=dict(color="white", width=0.5)),
        text=np.array(hover_texts)[mask],
        hovertemplate="%{text}<extra></extra>",
    ))

fig.update_layout(
    title="2D Vector Space — Sobhan's Knowledge Base (t-SNE)",
    xaxis_title="t-SNE dimension 1", yaxis_title="t-SNE dimension 2",
    legend_title="Category", width=850, height=600,
    template="plotly_dark",
    legend=dict(bgcolor="rgba(0,0,0,0.5)", bordercolor="white", borderwidth=1),
)
fig.show()
print("Tip: hover over any point to see its content. Click legend items to toggle categories.")

vectors type  : <class 'numpy.ndarray'>
vectors shape : (53, 1536)
vectors dtype : float64
NaN present   : False
Inf present   : False

Categories in store   : ['career', 'education', 'expertise', 'youtube']
Categories in COLORS  : ['career', 'expertise', 'education', 'youtube']

n_samples=53, using perplexity=10

Running t-SNE (2D)... this takes a few seconds


Tip: hover over any point to see its content. Click legend items to toggle categories.


In [12]:
# ── 3D t-SNE ──────────────────────────────────────────────────────────────────
print("Running t-SNE (3D)... this takes a few seconds")
tsne_3d    = TSNE(n_components=3, random_state=42, perplexity=10)
reduced_3d = tsne_3d.fit_transform(vectors)   # shape: (43, 3)

fig = go.Figure()
for category in CATEGORIES:
    mask = np.array([c == category for c in categories])
    fig.add_trace(go.Scatter3d(
        x=reduced_3d[mask, 0], y=reduced_3d[mask, 1], z=reduced_3d[mask, 2],
        mode="markers",
        name=category.title(),
        marker=dict(size=5, color=CATEGORY_COLORS[category], opacity=0.9),
        text=np.array(hover_texts)[mask],
        hovertemplate="%{text}<extra></extra>",
    ))

fig.update_layout(
    title="3D Vector Space — Sobhan's Knowledge Base (t-SNE)",
    scene=dict(xaxis_title="dim 1", yaxis_title="dim 2", zaxis_title="dim 3",
               bgcolor="#111111"),
    legend_title="Category", width=900, height=700,
    template="plotly_dark",
    legend=dict(bgcolor="rgba(0,0,0,0.5)", bordercolor="white", borderwidth=1),
)
fig.show()
print("Drag to rotate the cube. Scroll to zoom. Notice how same-category chunks cluster.")

Running t-SNE (3D)... this takes a few seconds


Drag to rotate the cube. Scroll to zoom. Notice how same-category chunks cluster.


### Where does a query land in vector space?

When a user asks a question, we embed it with the **same model** and find the nearest vectors.
Let's visualise this — embed a few real questions and plot them as **stars** on the 2D map.

A well-trained embedding model places the query star *inside or near the cluster* of documents
that contain the answer. This is the entire mechanism behind RAG retrieval.

In [13]:
QUERIES = [
    ("What is Sobhan's UX design philosophy?",   "#FFFFFF"),   # white — should land near expertise
    ("What companies has Sobhan worked at?",      "#FF00FF"),   # magenta — should land near career
    ("What are Sobhan's academic qualifications?","#00FFFF"),   # cyan — should land near education
]

print("Embedding queries...")
query_vecs = np.array([
    openai.embeddings.create(model=EMBEDDING_MODEL, input=[q]).data[0].embedding
    for q, _ in QUERIES
])

# t-SNE must see ALL points at once to produce a consistent layout
# (we cannot project queries independently into an existing t-SNE space)
all_vecs = np.vstack([vectors, query_vecs])      # (43+3, 1536)
print("Running t-SNE on documents + queries together...")
tsne_q  = TSNE(n_components=2, random_state=42, perplexity=10)
all_2d  = tsne_q.fit_transform(all_vecs)         # (46, 2)

doc_2d = all_2d[:len(vectors)]    # first 43 = document chunks
q_2d   = all_2d[len(vectors):]   # last  3  = query points

fig = go.Figure()

# Document chunks
for category in CATEGORIES:
    mask = np.array([c == category for c in categories])
    fig.add_trace(go.Scatter(
        x=doc_2d[mask, 0], y=doc_2d[mask, 1],
        mode="markers", name=category.title(),
        marker=dict(size=9, color=CATEGORY_COLORS[category], opacity=0.75,
                    line=dict(color="white", width=0.5)),
        text=np.array(hover_texts)[mask],
        hovertemplate="%{text}<extra></extra>",
    ))

# Query stars
for i, (query, color) in enumerate(QUERIES):
    fig.add_trace(go.Scatter(
        x=[q_2d[i, 0]], y=[q_2d[i, 1]],
        mode="markers+text",
        name=f"❓ {query[:40]}…",
        marker=dict(size=20, color=color, symbol="star",
                    line=dict(color="black", width=1.5)),
        text=[f"❓"], textposition="top center",
        textfont=dict(color=color, size=12),
        hovertemplate=f"<b>QUERY:</b> {query}<extra></extra>",
    ))

fig.update_layout(
    title="2D Vector Space with Query Points (⭐)",
    xaxis_title="t-SNE dim 1", yaxis_title="t-SNE dim 2",
    legend_title="Legend", width=900, height=650,
    template="plotly_dark",
    legend=dict(bgcolor="rgba(0,0,0,0.5)", bordercolor="white", borderwidth=1),
)
fig.show()
print("\n⭐ Stars show where each query lands in the semantic space.")
print("A good embedding places each query NEAR the cluster of documents that answer it.")

Embedding queries...
Running t-SNE on documents + queries together...



⭐ Stars show where each query lands in the semantic space.
A good embedding places each query NEAR the cluster of documents that answer it.


### Live retrieval — highlight which chunks get returned

Now let's run an actual ChromaDB similarity search and highlight the retrieved chunks on the map.
This is **exactly** what `agents/knowledge_base_agent.py` does on every request.

In [14]:
DEMO_QUESTION = "What did Sobhan achieve at Elisity?"
TOP_K         = 5

# Embed the question and search ChromaDB
print(f"Query: '{DEMO_QUESTION}'\n")
query_vector = openai.embeddings.create(
    model=EMBEDDING_MODEL, input=[DEMO_QUESTION]
).data[0].embedding

results = collection.query(
    query_embeddings=[query_vector],
    n_results=TOP_K,
    include=["documents", "metadatas", "distances"],
)

retrieved_texts = set(results["documents"][0])

print(f"Top {TOP_K} retrieved chunks:\n" + "─"*60)
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0]
), 1):
    sim = round(1 - dist, 3)
    print(f"\n[{i}] Source     : {meta['source']}  ({meta['category']})")
    print(f"    Similarity : {sim:.3f}")
    print(f"    Preview    : {doc[:200].strip()}...")

Query: 'What did Sobhan achieve at Elisity?'

Top 5 retrieved chunks:
────────────────────────────────────────────────────────────

[1] Source     : elisity  (career)
    Similarity : 0.555
    Preview    : # Head of UX & UI Engineering — Elisity Inc

## Role Overview
Sobhan Dutta served as Head of UX & UI Engineering at Elisity Inc in San Jose, CA from February 2020 to January 2022. He led UX and UI eng...

[2] Source     : leadership  (expertise)
    Similarity : 0.527
    Preview    : X & UI team at Elisity Inc as Head of UX & UI Engineering
- Led globally distributed teams at Nuance (Microsoft) across multiple time zones
- Hired, mentored, and grew high-performing designers and en...

[3] Source     : leadership  (expertise)
    Similarity : 0.468
    Preview    : less of team geography
- Established communication rhythms and processes for async collaboration

## Strategic Leadership vs Hands-On Execution
Sobhan is comfortable operating at both strategic and ex...

[4] Source     :

In [15]:
# Visualise the retrieved chunks highlighted on the 2D map
is_retrieved = np.array([t in retrieved_texts for t in doc_texts])

# Re-run t-SNE including the demo query point for a consistent layout
demo_query_vec = np.array([query_vector])
all_vecs_demo  = np.vstack([vectors, demo_query_vec])
print("Running t-SNE for retrieval visualisation...")
all_2d_demo    = TSNE(n_components=2, random_state=42, perplexity=10).fit_transform(all_vecs_demo)
doc_2d_demo    = all_2d_demo[:len(vectors)]
q_2d_demo      = all_2d_demo[len(vectors):]

fig = go.Figure()

# Non-retrieved docs (dim)
for category in CATEGORIES:
    mask = np.array([c == category for c in categories]) & ~is_retrieved
    if mask.any():
        fig.add_trace(go.Scatter(
            x=doc_2d_demo[mask, 0], y=doc_2d_demo[mask, 1],
            mode="markers", name=f"{category.title()} (not retrieved)",
            marker=dict(size=8, color=CATEGORY_COLORS[category], opacity=0.2),
            text=np.array(hover_texts)[mask],
            hovertemplate="%{text}<extra></extra>",
        ))

# Retrieved chunks (bright diamonds)
if is_retrieved.any():
    retrieved_labels = [
        f"{sources[i]} — sim: {round(1 - results['distances'][0][list(retrieved_texts).index(doc_texts[i]) if doc_texts[i] in retrieved_texts else 0], 3)}"
        if doc_texts[i] in retrieved_texts else ""
        for i in range(len(doc_texts))
    ]
    fig.add_trace(go.Scatter(
        x=doc_2d_demo[is_retrieved, 0], y=doc_2d_demo[is_retrieved, 1],
        mode="markers", name="✅ Retrieved chunks",
        marker=dict(size=18, color="#FACC15", symbol="diamond",
                    line=dict(color="white", width=1.5), opacity=1.0),
        text=np.array(hover_texts)[is_retrieved],
        hovertemplate="%{text}<extra></extra>",
    ))

# Query star
fig.add_trace(go.Scatter(
    x=[q_2d_demo[0, 0]], y=[q_2d_demo[0, 1]],
    mode="markers+text", name="❓ Query",
    marker=dict(size=22, color="#FF3B3B", symbol="star",
                line=dict(color="white", width=2)),
    text=["❓"], textposition="top center",
    hovertemplate=f"<b>QUERY:</b> {DEMO_QUESTION}<extra></extra>",
))

fig.update_layout(
    title=f"Retrieval: '{DEMO_QUESTION}'",
    xaxis_title="t-SNE dim 1", yaxis_title="t-SNE dim 2",
    legend_title="Legend", width=900, height=650,
    template="plotly_dark",
    legend=dict(bgcolor="rgba(0,0,0,0.5)", bordercolor="white", borderwidth=1),
)
fig.show()
print(f"\n🔴 Red star    = query")
print(f"🟡 Diamonds   = top-{TOP_K} retrieved chunks sent to Claude")
print(f"⬜ Dim points  = not retrieved this time")

Running t-SNE for retrieval visualisation...



🔴 Red star    = query
🟡 Diamonds   = top-5 retrieved chunks sent to Claude
⬜ Dim points  = not retrieved this time


---
## Summary

Here is the complete RAG pipeline you explored in this notebook:

```
8 Markdown files (knowledge_base/)
  │
  ├─ [Part A] Load: read text + tag with category (career/expertise/education)
  │    → 8 Document objects
  │
  ├─ [Part A] Chunk: sliding window (600 chars, 100 overlap)
  │    → 43 overlapping chunks
  │
  ├─ [Part B] Embed: OpenAI text-embedding-3-small
  │    → 43 × 1,536-dimensional vectors
  │
  ├─ [Part B] Persist in ChromaDB (vector_store/)
  │    → searchable by cosine similarity in milliseconds
  │
  └─ [Part C] Visualise with t-SNE
       → 2D map, 3D map, query star placement, retrieval highlight
```

### Key takeaways

| Concept | Rule of thumb |
|---|---|
| Chunk size | 500–800 chars for focused factual Q&A |
| Chunk overlap | 15–20% of chunk size prevents boundary losses |
| Embedding model | Same model at ingest AND query time — mixing models breaks retrieval |
| Retrieval K | Start at 5; increase if answers feel incomplete |
| t-SNE perplexity | Use low (5–15) for small datasets (<100 points) |
| Cosine similarity | Ranges -1 to 1; >0.7 = strongly related for this model |

### How this connects to the rest of the project

```
data/ingest_kb.py              ← builds the vector_store/ (run once)
agents/knowledge_base_agent.py ← embed → search → answer (runs on every query)
orchestrator.py                ← calls KnowledgeBaseAgent via tool: query_knowledge_base
tools/definitions.py           ← tells the LLM when to use query_knowledge_base
```

### Try changing things

- Change `DEMO_QUESTION` to any question about Sobhan and re-run the retrieval cells
- Change `TOP_K` from 5 to 10 — see more chunks highlighted
- Add a new `.md` file to `knowledge_base/`, re-run `data/ingest_kb.py`, then re-run this notebook
- Change `CHUNK_SIZE` in `data/ingest_kb.py` and compare chunk distributions